# Formula schemas — JSON and code (proof)

Scratch inspection only. Paths under `docs/proofs/multi-template-registry/` are **not** registered in the manifest yet.

Shows:

1. Proposed **registry** records (`formula_templates_scratch.jsonl`)
2. Legacy **formula_schemes** export (from republic_clean xlsx → JSON)
3. How a thin loader would look in code (read-only sketch; not wired into STEPS)


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()  # repo root when cwd is notebooks/
PROOF = ROOT / "docs" / "proofs" / "multi-template-registry"
print("PROOF", PROOF, "exists", PROOF.is_dir())
print("files:", sorted(p.name for p in PROOF.glob("*")))


## 1. Registry scratch JSONL (proposed `formula_templates`)


In [ ]:
registry_path = PROOF / "formula_templates_scratch.jsonl"
registry = [
    json.loads(line)
    for line in registry_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"{len(registry)} records\n")
for tid in (
    "opening_requeste_18c",
    "goedgevonden_decision_opener",
    "appointment_vergadering",
    "appointment_rekenkamer",
):
    row = next(r for r in registry if r["template_id"] == tid)
    print(json.dumps(row, ensure_ascii=False, indent=2))
    print("---")


In [ ]:
pd.DataFrame(registry)[
    [
        "template_id",
        "role",
        "doc_flag",
        "period",
        "threshold",
        "nlp_start_marker",
        "outcome_signal",
        "role_depends_on_place",
        "discriminators",
    ]
]


## 2. Legacy `formula_schemes` (xlsx → JSON)

Proof copy: `formula_schemes_sample.json`.  
Note: `preceded by: "<START>"` is an **NLP artefact**, not archival text.


In [ ]:
schemes_path = PROOF / "formula_schemes_sample.json"
schemes = json.loads(schemes_path.read_text(encoding="utf-8"))
print(f"{len(schemes)} scheme rows\n")
for name in (
    "ontfangen_start",
    "requeste_start",
    "requeste",
    "goedgevonden",
    "geen_resolutie",
):
    row = next(r for r in schemes if r.get("phrasename") == name)
    print(json.dumps(row, ensure_ascii=False, indent=2))
    print("---")


In [ ]:
schemes_df = pd.DataFrame(schemes)
schemes_df[
    [
        "phrasename",
        "proposition_type",
        "canonic_formula",
        "function",
        "intention",
        "preceded by",
        "followed by",
        "search phrase",
    ]
]


## 3. Code sketch — load registry (not wired into STEPS)

After the gate, this would live in `formulation/boundary/templates.py` and read via `data_io.load("formula_templates")`.


In [ ]:
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class FormulaTemplate:
    template_id: str
    role: str
    tokens: tuple[str, ...]
    threshold: float | None
    doc_flag: str | None
    zones_hypothesized: tuple[str, ...]
    nlp_start_marker: bool
    outcome_signal: bool
    role_depends_on_place: bool
    discriminators: tuple[str, ...]
    raw: dict[str, Any]


def load_registry_from_jsonl(path: Path) -> dict[str, FormulaTemplate]:
    out: dict[str, FormulaTemplate] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        tid = str(row["template_id"])
        out[tid] = FormulaTemplate(
            template_id=tid,
            role=str(row.get("role") or ""),
            tokens=tuple(row.get("tokens") or ()),
            threshold=row.get("threshold"),
            doc_flag=row.get("doc_flag"),
            zones_hypothesized=tuple(row.get("zones_hypothesized") or ()),
            nlp_start_marker=bool(row.get("nlp_start_marker")),
            outcome_signal=bool(row.get("outcome_signal")),
            role_depends_on_place=bool(row.get("role_depends_on_place")),
            discriminators=tuple(row.get("discriminators") or ()),
            raw=row,
        )
    return out


def framing_templates(reg: dict[str, FormulaTemplate]) -> list[FormulaTemplate]:
    return [
        t
        for t in reg.values()
        if t.role == "framing" and t.tokens and t.threshold is not None
    ]


reg = load_registry_from_jsonl(registry_path)
print("ids:", sorted(reg))
opening = reg["opening_requeste_18c"]
print("framing default:", opening.template_id, opening.tokens, "cut", opening.threshold)
print("goedgevonden outcome_signal?", reg["goedgevonden_decision_opener"].outcome_signal)
print("requeste_start nlp_start_marker?", reg["requeste_start"].nlp_start_marker)
print("appointment discriminators:", reg["appointment_rekenkamer"].discriminators)
print("framing_templates:", [t.template_id for t in framing_templates(reg)])


## 4. Join scheme ↔ registry by `doc_flag` / `phrasename`


In [ ]:
by_phrase = {r["phrasename"]: r for r in schemes if r.get("phrasename")}
joined = []
for t in registry:
    flag = t.get("doc_flag")
    scheme = by_phrase.get(flag) if flag else None
    joined.append(
        {
            "template_id": t["template_id"],
            "role": t["role"],
            "doc_flag": flag,
            "scheme_function": (scheme or {}).get("function"),
            "scheme_intention": (scheme or {}).get("intention"),
            "scheme_preceded_by": (scheme or {}).get("preceded by"),
            "scheme_canonic": (scheme or {}).get("canonic_formula"),
            "registry_outcome_signal": t.get("outcome_signal"),
            "registry_nlp_start": t.get("nlp_start_marker"),
        }
    )
pd.DataFrame(joined)
